In [ ]:
# Mesh retrieved from: https://github.com/spbu-math-cs/Riemannian-Gaussian-Processes?tab=readme-ov-file

In [ ]:
from firedrake import Mesh, FunctionSpace
from manifold_matern.utils import construct_mesh_graph

import numpy as np

import networkx as nx
import time
import os


# Extract Mesh

In [ ]:
def construct_ground_truth(mesh):

    mesh_graph = construct_mesh_graph(mesh)

    geodesics = nx.shortest_path_length(mesh_graph, source=0, weight='weight')

    N = mesh.num_vertices()

    ground_truth = np.zeros((N))
    period = 2*np.pi / 0.3 * 2
    for i in range(N):
        ground_truth[i] = 2 * np.sin(geodesics.get(i) * period + 0.3)

    return ground_truth

In [3]:
mesh = Mesh('resources/meshes/dragon_connected.msh', dim=3)

print('Constructing ground truth. It may take a while')

ground_truth = construct_ground_truth(mesh)

V = FunctionSpace(mesh, "Lagrange", 1)


Constructing ground truth. It may take a while


# Computing full geodesic matrix

In [ ]:
# Run this in Firedrake environment

print("Starting Geodesic Distance Matrix Computation (Upper Triangle Only)")
print("=" * 70)

# Configuration
N_VERTICES_SUBSET = 100179  
OUTPUT_FILE = f'dragon_geodesic_matrix_{N_VERTICES_SUBSET}_new.npy'
CHECKPOINT_FILE = f'dragon_geodesic_checkpoint_{N_VERTICES_SUBSET}_new.npz'
SAVE_INTERVAL = 5000  

# Load mesh and create graph
print("Loading mesh and constructing graph...")
mesh = Mesh('resources/meshes/dragon_connected.msh', dim=3)
vertices_full = mesh.coordinates.dat.data_ro.copy()
mesh_graph_full = construct_mesh_graph(mesh)

# Extract subset
vertices = vertices_full[:N_VERTICES_SUBSET]
n_vertices = len(vertices)

# Create subgraph with only these vertices
print(f"Creating subgraph for first {n_vertices:,} vertices...")
mesh_graph = mesh_graph_full.subgraph(range(n_vertices)).copy()

print(f"Mesh loaded: {len(vertices_full):,} total vertices")
print(f"Using subset: {n_vertices:,} vertices")
print(f"Subgraph created: {mesh_graph.number_of_nodes():,} nodes, {mesh_graph.number_of_edges():,} edges")
print(f"Memory required: {n_vertices * n_vertices * 4 / 1e9:.1f} GB")

# Check for existing checkpoint
start_vertex = 0
start_time = time.time()

if os.path.exists(CHECKPOINT_FILE):
    print(f"\n Found checkpoint file: {CHECKPOINT_FILE}")
    try:
        checkpoint_data = np.load(CHECKPOINT_FILE)
        start_vertex = int(checkpoint_data['last_completed_vertex']) + 1
        checkpoint_start_time = float(checkpoint_data['start_time'])
        
        # Use existing start time if resuming
        start_time = checkpoint_start_time
        
        print(f"Resuming from vertex {start_vertex:,}/{n_vertices:,}")
        print(f"Previously elapsed: {checkpoint_data.get('elapsed_time', 0)/60:.1f} minutes")
        
        # Load existing matrix
        if os.path.exists(OUTPUT_FILE):
            print(f"Found existing matrix file: {OUTPUT_FILE}")
            geo_dist_full = np.memmap(OUTPUT_FILE,
                                      dtype=np.float32,
                                      mode='r+',
                                      shape=(n_vertices, n_vertices))
        else:
            print(f"Checkpoint exists but matrix file missing. Starting fresh.")
            start_vertex = 0
            geo_dist_full = np.memmap(OUTPUT_FILE,
                                      dtype=np.float32,
                                      mode='w+',
                                      shape=(n_vertices, n_vertices))
            geo_dist_full[:] = np.inf
            np.fill_diagonal(geo_dist_full, 0.0)
            geo_dist_full.flush()
            
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Starting fresh computation...")
        start_vertex = 0
        start_time = time.time()
        
        # Create new matrix
        geo_dist_full = np.memmap(OUTPUT_FILE,
                                  dtype=np.float32,
                                  mode='w+',
                                  shape=(n_vertices, n_vertices))
        geo_dist_full[:] = np.inf
        np.fill_diagonal(geo_dist_full, 0.0)
        geo_dist_full.flush()
else:
    print("No checkpoint found, starting fresh computation...")
    
    # Create new matrix
    geo_dist_full = np.memmap(OUTPUT_FILE,
                              dtype=np.float32,
                              mode='w+',
                              shape=(n_vertices, n_vertices))
    geo_dist_full[:] = np.inf
    np.fill_diagonal(geo_dist_full, 0.0)
    geo_dist_full.flush()
    print(f"Matrix initialized with shape {geo_dist_full.shape}")

print(f"\n Starting computation from vertex {start_vertex:,}")
print(f"Checkpoints will be saved every {SAVE_INTERVAL} vertices")
print(f"Computing UPPER TRIANGLE ONLY (will mirror at end)")
print("-" * 70)

# Computation loop
batch_size = 100
last_checkpoint_vertex = start_vertex - (start_vertex % SAVE_INTERVAL)

for batch_start in range(start_vertex, n_vertices, batch_size):
    batch_end = min(batch_start + batch_size, n_vertices)
    
    # Progress reporting
    elapsed = time.time() - start_time
    if batch_start > start_vertex:
        vertices_done = batch_start - start_vertex
        if vertices_done > 0:
            est_total = elapsed * (n_vertices - start_vertex) / vertices_done
            remaining = est_total - elapsed
            pct = batch_start / n_vertices * 100
            
            print(f"Vertex {batch_start:6,}/{n_vertices:,} ({pct:5.1f}%) - "
                  f"Elapsed: {elapsed/60:6.1f}m - ETA: {remaining/60:6.1f}m")
    
    # Compute batch - UPPER TRIANGLE ONLY
    for source in range(batch_start, batch_end):
        try:
            # Compute shortest paths from this source
            distances_dict = nx.shortest_path_length(mesh_graph, 
                                                     source=source, 
                                                     weight='weight')
            
            # Fill ONLY upper triangle (where target >= source)
            for target, dist in distances_dict.items():
                if target >= source:
                    geo_dist_full[source, target] = dist
                
        except Exception as e:
            print(f"Error at vertex {source}: {e}")
            # Fill with inf for failed vertices
            geo_dist_full[source, source:] = np.inf
            geo_dist_full[source, source] = 0.0
    
    # Flush to disk
    geo_dist_full.flush()
    
    # Save checkpoint periodically
    if batch_end >= last_checkpoint_vertex + SAVE_INTERVAL:
        elapsed_time = time.time() - start_time
        print(f"Saving checkpoint at vertex {batch_end-1:,}...")
        
        np.savez_compressed(CHECKPOINT_FILE,
                           last_completed_vertex=batch_end - 1,
                           total_vertices=n_vertices,
                           start_time=start_time,
                           elapsed_time=elapsed_time)
        
        last_checkpoint_vertex = batch_end - (batch_end % SAVE_INTERVAL)
        print(f"Checkpoint saved (elapsed: {elapsed_time/60:.1f} min)")

# Final flush
geo_dist_full.flush()
computation_time = time.time() - start_time

print("\n" + "=" * 70)
print(f"Upper triangle computation complete in {computation_time/60:.1f} minutes ({computation_time/3600:.2f} hours)")

In [ ]:

print("="*70)
print("FAST GEODESIC MATRIX MIRRORING")
print("="*70)

N_VERTICES = 100179
INPUT_FILE = f'dragon_geodesic_matrix_{N_VERTICES}_new.npy'

print(f"\n Loading matrix: {INPUT_FILE}")
geo_dist = np.memmap(INPUT_FILE,
                     dtype=np.float32,
                     mode='r+',  # Read-write mode
                     shape=(N_VERTICES, N_VERTICES))

print(f" Loaded: {geo_dist.shape}")

# ===== FAST BATCH TRANSPOSE MIRRORING =====
print("\n Mirroring upper triangle to lower (batch method)...")
print("="*70)

mirror_start = time.time()

# Process in large blocks for efficiency
block_size = 1000  

for i in range(0, N_VERTICES, block_size):
    i_end = min(i + block_size, N_VERTICES)
    
    # Only process blocks on or above the diagonal
    for j in range(i, N_VERTICES, block_size):
        j_end = min(j + block_size, N_VERTICES)
        
        # Read block from upper triangle
        upper_block = geo_dist[i:i_end, j:j_end].copy()
        
        # Write transposed to lower triangle
        # (Skip when i==j since that's the diagonal block)
        if i != j:
            geo_dist[j:j_end, i:i_end] = upper_block.T
        else:
            # For diagonal blocks, only copy upper triangle within the block
            for local_row in range(len(upper_block)):
                global_row = i + local_row
                if global_row + 1 < N_VERTICES:
                    # Copy rest of row to corresponding column
                    n_copy = min(i_end - global_row - 1, N_VERTICES - global_row - 1)
                    if n_copy > 0:
                        geo_dist[global_row+1:global_row+1+n_copy, global_row] = \
                            upper_block[local_row, local_row+1:local_row+1+n_copy]
    
    # Progress reporting
    pct = 100 * i_end / N_VERTICES
    elapsed = time.time() - mirror_start
    if i > 0:
        eta = elapsed * (N_VERTICES - i_end) / i_end / 60
        print(f"  Block {i:6,}-{i_end:6,} ({pct:5.1f}%) - Elapsed: {elapsed/60:.1f}m - ETA: {eta:.1f}m")
    else:
        print(f"  Block {i:6,}-{i_end:6,} ({pct:5.1f}%)")
    
    # Flush every few blocks
    if i % (block_size * 5) == 0:
        geo_dist.flush()

# Final flush
geo_dist.flush()

mirror_time = time.time() - mirror_start
print(f"\n Mirroring complete in {mirror_time/60:.1f} minutes ({mirror_time/3600:.2f} hours)")

# Quick verification
print("\n Quick verification...")
print("Checking symmetry on sample...")
sample_idx = [0, 100, 1000, 10000, 50000]
is_symmetric = True
for i in sample_idx:
    for j in sample_idx:
        if abs(geo_dist[i, j] - geo_dist[j, i]) > 1e-5:
            print(f"  Asymmetry at [{i},{j}]: {geo_dist[i,j]} vs {geo_dist[j,i]}")
            is_symmetric = False

if is_symmetric:
    print(" Sample check passed - matrix appears symmetric")

print("\n" + "="*70)
print("MIRRORING COMPLETE!")
print("="*70)

del geo_dist